### <span style="color:#FFA726">Popularity Baseline</span>

This notebook has three parts:

1. **Pre-Code Notes**: the plan, columns used, and sanity checks, before any code.
2. **Baseline Code**: building the popularity ranking and generating top-10 recommendations.
3. **Post-Code Sanity Checks**: verifying the baseline behaves as expected.




### <span style="color:#FFA726">1. Pre-Code Notes</span>

**What is this?** The simplest recommender: everyone gets the *same* top-10 list (the most-rated items overall). No personalization.

**Columns used:**
- `parent_asin`: to count how popular each item is
- `user_id`: to check what a user already rated, so we don't recommend it again
- `rating` is *not* used. Popularity here means "rated by the most people," not "highest average score."

**Steps:**

1. **Count:** using only `train`, count how many times each item was rated.
2. **Rank:** sort items from most-rated to least-rated.
3. **Recommend:** give every user the top-10 most popular items, skipping any 
   item that user already rated.
4. **Check:** before filtering, the top-10 list should be identical for every 
   user. That's expected, since nothing is personalized yet.

### <span style="color:#FFA726">2. Baseline Code</span>

In [1]:
import pandas as pd

train = pd.read_parquet('datasets/train.parquet')
validation = pd.read_parquet('datasets/validation.parquet')
test = pd.read_parquet('datasets/test.parquet')

print("train:", len(train))
print("validation:", len(validation))
print("test:", len(test))

train: 6126723
validation: 657203
test: 657203


In [2]:
item_counts = train['parent_asin'].value_counts()

top_10_items = item_counts.head(10).index.tolist()

print("Top 10 most popular items:")
print(item_counts.head(10))

Top 10 most popular items:
parent_asin
B00I3MQNWG    17509
B00RSGIVVO    15331
B003AZCYCE     9083
B01AB17IGQ     8836
B01J4SRJFW     8231
B00MR9UY8A     7827
B007SPQZMC     7444
B00X8UKOUK     7287
B009ZQC7MY     6821
B00RSGHX3Q     6527
Name: count, dtype: int64


In [3]:
def recommend_for_user(user_id, k=10):
    already_rated = set(train[train['user_id'] == user_id]['parent_asin'])
    recommendations = [item for item in item_counts.index if item not in already_rated][:k]
    return recommendations

sample_user = train['user_id'].iloc[0]
print("Sample user:", sample_user)
print("Recommendations:", recommend_for_user(sample_user))

Sample user: AE2222FRPDMNOMYOMCWIANTXP7UQ
Recommendations: ['B00I3MQNWG', 'B00RSGIVVO', 'B003AZCYCE', 'B01AB17IGQ', 'B01J4SRJFW', 'B00MR9UY8A', 'B007SPQZMC', 'B00X8UKOUK', 'B009ZQC7MY', 'B00RSGHX3Q']


In [4]:
sample_user = train['user_id'].iloc[0]
recs = recommend_for_user(sample_user)

print("Sample user:", sample_user)
print("\nRecommendations:")

for asin in recs:
    title_row = train[train['parent_asin'] == asin]['title']
    title = title_row.iloc[0] if len(title_row) > 0 and title_row.iloc[0] is not None else "(no title)"
    print(f"  {asin}  —  {title}")

Sample user: AE2222FRPDMNOMYOMCWIANTXP7UQ

Recommendations:
  B00I3MQNWG  —  nan
  B00RSGIVVO  —  nan
  B003AZCYCE  —  nan
  B01AB17IGQ  —  nan
  B01J4SRJFW  —  nan
  B00MR9UY8A  —  nan
  B007SPQZMC  —  nan
  B00X8UKOUK  —  nan
  B009ZQC7MY  —  nan
  B00RSGHX3Q  —  nan


In [5]:
check = train[train['parent_asin'].isin(recs)][['parent_asin', 'title', 'main_category']].drop_duplicates()
print(check)

     parent_asin title main_category
45    B00I3MQNWG   NaN   Prime Video
154   B009ZQC7MY   NaN   Prime Video
264   B007SPQZMC   NaN   Prime Video
275   B00RSGIVVO   NaN   Prime Video
277   B003AZCYCE   NaN   Prime Video
374   B01AB17IGQ   NaN   Prime Video
1340  B01J4SRJFW   NaN   Prime Video
1714  B00RSGHX3Q   NaN   Prime Video
1740  B00MR9UY8A   NaN   Prime Video
3291  B00X8UKOUK   NaN   Prime Video


**Note:** The top-10 most popular items are all from the **Prime Video** category, 
which has no `title` metadata (as seen in EDA — ~89% of Prime Video items are 
missing metadata). This is expected, not a bug, and will be noted as a limitation 
when doing the genre-level analysis later.

<span style="color:#1A237E">Prime Video content is a much 
lower-effort interaction than buying physical media, driving up its interaction 
counts. Since Prime Video is managed as a separate catalog without product 
metadata, most of the Popularity baseline's top items lack titles and genres,
directly limiting genre-level analysis coverage.</span>

In [6]:
top_pool = item_counts.head(50).index.tolist()

user_rated = train.groupby('user_id')['parent_asin'].apply(set)

def get_top10(rated_items):
    return [item for item in top_pool if item not in rated_items][:10]

recommendations = user_rated.apply(get_top10)

print("Total users:", len(recommendations))
print(recommendations.head())

Total users: 657203
user_id
AE2222FRPDMNOMYOMCWIANTXP7UQ    [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...
AE22236AFRRSMQIKGG7TPTB75QEA    [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...
AE222H3FGXWLHRFUMGMS2RR57NDQ    [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...
AE223AAG2OHFGX6BTTD3HGJZCMDQ    [B00RSGIVVO, B003AZCYCE, B01AB17IGQ, B01J4SRJF...
AE223CNRS5AQOKRWCFPRBUNJ5T2Q    [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...
Name: parent_asin, dtype: object


<span style="color:#1A237E">The first three sample users 
received the identical top-10 list, since none had rated any of the globally 
most popular items. The fourth user's list correctly excluded the #1 most popular 
item (`B00I3MQNWG`), since they had already rated it in the training data, 
confirming the exclusion logic behaves as intended.</span>

### <span style="color:#FFA726">3. Post-Code Sanity Checks</span>

We verify that the baseline behaves as expected: the raw (unfiltered) top-10 
ranking is identical for every user, since nothing is personalized before the 
exclusion step.

In [8]:
recommendations_df = recommendations.reset_index()
recommendations_df.columns = ['user_id', 'top10']

# Sanity check: the raw top-10 (before exclusion) is the same for everyone by construction,
# since item_counts is computed once from train and shared across all users.
raw_top10 = item_counts.head(10).index.tolist()
print("Raw top-10 (before exclusion):", raw_top10)

# Confirm total row/user counts match
print("\nTotal users with recommendations:", len(recommendations_df))
print("Expected:", train['user_id'].nunique())
print("Match:", len(recommendations_df) == train['user_id'].nunique())

# Confirm every user got exactly 10 recommendations
rec_lengths = recommendations_df['top10'].apply(len)
print("\nUsers with fewer than 10 recommendations:", (rec_lengths < 10).sum())
print("Users with exactly 10 recommendations:", (rec_lengths == 10).sum())

Raw top-10 (before exclusion): ['B00I3MQNWG', 'B00RSGIVVO', 'B003AZCYCE', 'B01AB17IGQ', 'B01J4SRJFW', 'B00MR9UY8A', 'B007SPQZMC', 'B00X8UKOUK', 'B009ZQC7MY', 'B00RSGHX3Q']

Total users with recommendations: 657203
Expected: 657203
Match: True

Users with fewer than 10 recommendations: 0
Users with exactly 10 recommendations: 657203


<span style="color:#1A237E">All 657,203 users received exactly 10 recommendations, 
with none falling short, confirming the popularity buffer (top-50 pool) was large 
enough to guarantee 10 recommendations per user even after excluding already-rated 
items.</span>

##### <span style="color:#BA68C8">Saving the recommendations</span>

We save the per-user top-10 recommendations to `datasets/` as a parquet file, 
so we don't need to recompute them later during the Evaluation step (NDCG, 
Recall, Coverage).

In [10]:
recommendations_df = recommendations.reset_index()
recommendations_df.columns = ['user_id', 'top10']
recommendations_df.to_parquet('datasets/popularity_recommendations.parquet', index=False)

print("Saved.")
print(recommendations_df.head())


Saved.
                        user_id  \
0  AE2222FRPDMNOMYOMCWIANTXP7UQ   
1  AE22236AFRRSMQIKGG7TPTB75QEA   
2  AE222H3FGXWLHRFUMGMS2RR57NDQ   
3  AE223AAG2OHFGX6BTTD3HGJZCMDQ   
4  AE223CNRS5AQOKRWCFPRBUNJ5T2Q   

                                               top10  
0  [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...  
1  [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...  
2  [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...  
3  [B00RSGIVVO, B003AZCYCE, B01AB17IGQ, B01J4SRJF...  
4  [B00I3MQNWG, B00RSGIVVO, B003AZCYCE, B01AB17IG...  
